# 🌡️ İklim Değişikliği Trend ve SQMK Kırılma Analizi
**Mustafa Kerem Öztaş** | Pamukkale Üniversitesi Fen Bilimleri Enstitüsü İNŞAAT MÜHENDİSLİĞİ

Bu notebook, meteorolojik veriler üzerinde **TFPW Pre-whitening**, **Sen's Slope** ve **Sequential Mann-Kendall (SQMK)** analizlerini yürütür.

In [ ]:
# 1. Gerekli Kütüphanelerin Yüklenmesi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Lütfen analiz edilecek MGM Excel dosyasını seçip yükleyin:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# 2. Veri Temizleme Fonksiyonu
def load_and_clean_data(file_path, station_row_idx=1, data_rows=16):
    df_raw = pd.read_excel(file_path, skiprows=4)
    df_station = df_raw.iloc[station_row_idx:station_row_idx+data_rows, 1:14].copy()
    df_station.columns = ['Yıl', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
    
    df_station['Yıl'] = pd.to_numeric(df_station['Yıl'])
    for month in range(1, 13):
        df_station[str(month)] = pd.to_numeric(df_station[str(month)])
        
    years = df_station['Yıl'].values
    annual_max = df_station.iloc[:, 1:13].max(axis=1).values
    return years, annual_max

# 3. Sen's Slope Fonksiyonu
def calc_sens_slope(x):
    n = len(x)
    slopes = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            slopes.append((x[j] - x[i]) / (j - i))
    return np.median(slopes)

# 4. TFPW Pre-Whitening Fonksiyonu
def tfpw_prewhitening(x, beta):
    n = len(x)
    t = np.arange(1, n + 1)
    y = x - beta * t
    y_mean = np.mean(y)
    r1 = np.sum((y[:-1] - y_mean) * (y[1:] - y_mean)) / np.sum((y - y_mean) ** 2)
    y_prime = y[1:] - r1 * y[:-1]
    x_tfpw = y_prime + beta * t[1:]
    return x_tfpw, r1

# 5. SQMK Fonksiyonu
def sequential_mann_kendall(x):
    n = len(x)
    def calculate_u(series):
        u = np.zeros(n)
        t_stat = 0
        for k in range(1, n):
            n_k = np.sum(series[:k] < series[k])
            t_stat += n_k
            E_t = (k + 1) * k / 4.0
            Var_t = (k + 1) * k * (2 * (k + 1) + 5) / 72.0
            u[k] = (t_stat - E_t) / np.sqrt(Var_t)
        return u

    uf = calculate_u(x)
    ub = calculate_u(x[::-1])[::-1] * -1.0
    return uf, ub

# 6. Analizi Çalıştırma ve Görselleştirme
years, temp_series = load_and_clean_data(file_name)
beta = calc_sens_slope(temp_series)
tfpw_series, r1 = tfpw_prewhitening(temp_series, beta)
uf, ub = sequential_mann_kendall(tfpw_series)

# TFPW nedeniyle 1 yıl azalan zaman dizisini hizalama
years_tfpw = years[1:]

# Grafik
plt.figure(figsize=(10, 5), dpi=150)
plt.plot(years_tfpw, uf, 'r-o', label='UF (İleri İstatistik)', linewidth=2)
plt.plot(years_tfpw, ub, 'b--s', label='UB (Geri İstatistik)', linewidth=2)
plt.axhline(y=1.96, color='k', linestyle='--', alpha=0.7, label='+1.96 (%95 Güven Sınırı)')
plt.axhline(y=-1.96, color='k', linestyle='--', alpha=0.7, label='-1.96 (%95 Güven Sınırı)')
plt.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

plt.title("Yıllık Maks. Sıcaklık SQMK Kırılma Analizi (Google Colab)", fontsize=12)
plt.xlabel("Yıllar", fontsize=10)
plt.ylabel("Z İstatistiği u(t)", fontsize=10)
plt.legend(loc='best')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print(f"\n--- HESAPLAMA SONUÇLARI ---")
print(f"Sen's Slope Eğimi (Derece Değişim Oranı): {beta:.4f} °C/yıl")
print(f"Lag-1 Otokorelasyon (r1): {r1:.4f}")